<a href="https://colab.research.google.com/github/r1shi-j/coa122-coursework/blob/main/122_coursework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import subscriptionManager as smSL
import feedbackManager as fm
from datetime import datetime

# Check Feedback Manager Works.
# feedback_list = fm.load_feedback()
# print(feedback_list)  # Should print the list of feedback records.
# fm.add_feedback("dft01", 4, "Not bad", "Game_Feedback.txt")

# # Check Subscription Manager Works.
subscriptions = smSL.load_subscriptions()
print(smSL.check_subscription("lbro", subscriptions))
# # Should return True or False based on the current date.
print(smSL.get_rental_limit("Basic"))  # Should return 2.
print(smSL.get_rental_limit("Premium"))  # Should return 7.

True
2
7


In [12]:
# fm.add_feedback("hi", 999, "Excel", "Game_Feedback.txt")
# feedback_list = fm.load_feedback(file_name="Game_Feedback.txt")
# print(feedback_list)

### System Skeleton
Show main menu
- check availability of tools
  - search for tools
- rent tools to customers
  - must have valid subscription
  - must have not reached their renting limit
  - tool must be available (and a valid tool)
  - return tools
  - search for tools
- maintain a database of all tools
  - search for tools
- pruning
  - reads Tool_Feedback to understand which tools aren't popular
-----------------------------------------------------
_Files to use_

`Tool_Info.txt`
- stores ToolID,Name,Size,PurchasePrice,PurchaseDate
- all tools
- read-only
  - list all tools
  - search tools given a search term
  - check if tool exists

`Tool_Feedback.txt`
- stores ToolID,Rating,Comments
- logs customer feedback about the tool
- read-append
  - log tool feedback
  - find the least popular tools

`Subscription_Info.txt`
- stores CustomerID,SubscriptionType,StartDate,EndDate
- all customers with a subscription
- read-only
  - check if customer has a subscription
  - check if customer has reached their renting limit

`Rental_History.txt`
- stores ToolID,CustomerID,RentalDate,ReturnDate
- logs all rentals
- read-append
  - check if tool is available
  - log a tool rental
  - log a tool return

`Booking.txt`
- stores DIYClass,BookingDate,Time
- all DIY class bookings?
- read-only?

-----------------------------------------------------

Make dummy data
- generate_id function for 4-char UUID

In [13]:
def loadTools():
    tools = []
    with open("toolInfo.txt", "r") as file:
        next(file)

        for line in file:
            line = line.strip()
            if line:
                id, name, size, pprice, pdate = line.split(',')
                tools.append({'id': id, 'name': name, 'size': size, 'purchasePrice': int(pprice), 'purchaseDate': datetime.strptime(pdate, '%Y-%m-%d')})
    return tools


In [14]:
print(loadTools())

[{'id': '1', 'name': 'Toolbox', 'size': 'Small', 'purchasePrice': 20, 'purchaseDate': datetime.datetime(2026, 8, 1, 0, 0)}, {'id': '2', 'name': 'Bulldozer', 'size': 'Large', 'purchasePrice': 3000, 'purchaseDate': datetime.datetime(2026, 8, 2, 0, 0)}, {'id': '3', 'name': 'Stump Grinder', 'size': 'Medium', 'purchasePrice': 1000, 'purchaseDate': datetime.datetime(2026, 8, 3, 0, 0)}]


In [25]:
def checkCurrentRentals(rentingCustomerID):
    count = 0
    with open("Rental_History.txt", "r") as file:
        next(file)

        for line in file:
            line = line.strip()
            if line:
                toolID, rentalDate, returnDate, customerID = line.split(',')
                # tools.append({'id': id, 'name': name, 'size': size, 'purchasePrice': int(pprice), 'purchaseDate': datetime.strptime(pdate, '%Y-%m-%d')})
                if customerID == rentingCustomerID and datetime.strptime(rentalDate, '%Y-%m-%d') < datetime.now():# < datetime.strptime(returnDate, '%Y-%m-%d'):
                    count += 1
    return count

In [29]:
print("Welcome to the Tool Rental and DIY booking management system")
print("Please select from the following options")
opt = input("a) Check the availibility of tools\nr) Rent tools out to a customer\nd) View a database of tools\n")
if opt == 'a':
    print("option a")
elif opt == 'r':
    customerID = input("Which customer wishes to rent a tool? (Enter their id)\n")

    subscriptions = smSL.load_subscriptions()
    hasSubscription = smSL.check_subscription(customerID, subscriptions)
    subscriptionPlan = subscriptions[customerID]['SubscriptionType']

    if hasSubscription:
        print("Subscription confirmed")

        # check if they have reached their renting limit
        currentRentalCount = checkCurrentRentals(customerID)
        if currentRentalCount < smSL.get_rental_limit(subscriptionPlan):
            print("They have not reached their renting limit")

            # check if tool available

            toolID = str(input("Enter the tool id they wish to rent out."))
            tools = loadTools()
            tool = None
            for t in tools:
                if t['id'] == toolID:
                    print("Valid tool")
                    tool = t
            if not tool:
                print("Invalid tool id")
            print(tool['name'])

        else:
            print("They have reached their renting limit")
    else:
        print(f"{customerID} does not have an onging subscription, so they will not be able to rent out any tools.")


elif opt == 'd':
    print("Here are all of the tools we have: ")

    tools = loadTools()
    for tool in tools:
        print(f"• {tool['name']}")

else:
    print("Invalid option")

Welcome to the Tool Rental and DIY booking management system
Please select from the following options
a) Check the availibility of tools
r) Rent tools out to a customer
d) View a database of tools
d
Here are all of the tools we have: 
• Toolbox
• Bulldozer
• Stump Grinder


1.1 Overview
Your task is to develop a simple tool rental and DIY booking management system. The system should enable
a store manager to manage tool rentals based on customer subscriptions and offer DIY classes.
The
manager should be able to:
- check the availability of tools,
- rent them out to customers
- maintain a
database of all tools in the inventory.

For each tool, the following information should be stored: name, size,
purchase price, purchase date, and a unique ID number to distinguish different copies of the same tool.

In [16]:
import secrets
import string

def generate_id(length=4):
    alphabet = string.ascii_letters + string.digits
    return ''.join(secrets.choice(alphabet) for _ in range(length))

### TODO
# Cut lines to 80 chars max, via colab divider line